In [13]:
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, TensorDataset
from torchvision.datasets import FashionMNIST

SEED, BATCH = 42, 128
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(SEED)

train_raw = FashionMNIST(ROOT / "data", train=True, download=True)
test_raw = FashionMNIST(ROOT / "data", train=False, download=True)
images = torch.cat([train_raw.data, test_raw.data]).unsqueeze(1).float() / 255.0
labels = torch.cat([train_raw.targets, test_raw.targets])
classes = train_raw.classes

with np.load(ROOT / "results/metrics/fashion_mnist_seed42.npz") as f:
    splits = {k: TensorDataset(images[f[k]], labels[f[k]]) for k in ("train", "val", "test")}

def get_loader(name):
    return DataLoader(splits[name], BATCH, shuffle=(name == "train"), generator=torch.Generator().manual_seed(SEED))

display(pd.DataFrame({k: splits[k].tensors[1].bincount(minlength=10).numpy() for k in ("train", "val", "test")}, index=classes))

,train,val,test
T-shirt/top,5600,700,700
Trouser,5600,700,700
Pullover,5600,700,700
Dress,5600,700,700
Coat,5600,700,700
Sandal,5600,700,700
Shirt,5600,700,700
Sneaker,5600,700,700
Bag,5600,700,700
Ankle boot,5600,700,700


In [14]:
from torch import nn

class DeterministicAE(nn.Module):
    def __init__(self, latent=64):
        super().__init__()
        self.encoder = nn.Sequential(nn.Flatten(), nn.Linear(784, 256), nn.ReLU(), nn.Linear(256, latent), nn.ReLU())
        self.decoder = nn.Sequential(nn.Linear(latent, 256), nn.ReLU(), nn.Linear(256, 784), nn.Sigmoid(), nn.Unflatten(1, (1, 28, 28)))
    def forward(self, x):
        return self.decoder(self.encoder(x))

class VariationalAE(nn.Module):
    def __init__(self, latent=64):
        super().__init__()
        self.encoder = nn.Sequential(nn.Flatten(), nn.Linear(784, 256), nn.ReLU())
        self.mean, self.logvar = nn.Linear(256, latent), nn.Linear(256, latent)
        self.decoder = nn.Sequential(nn.Linear(latent, 256), nn.ReLU(), nn.Linear(256, 784), nn.Unflatten(1, (1, 28, 28)))
    def encode(self, x):
        h = self.encoder(x)
        return self.mean(h), self.logvar(h)
    def forward(self, x):
        return self.decoder(self.encode(x)[0]).sigmoid()

class VanillaGAN(nn.Module):
    def __init__(self, z_dim=100):
        super().__init__()
        self.generator = nn.Sequential(nn.Linear(z_dim, 256), nn.ReLU(), nn.Linear(256, 512), nn.ReLU(), nn.Linear(512, 784), nn.Tanh(), nn.Unflatten(1, (1, 28, 28)))
        self.discriminator = nn.Sequential(nn.Flatten(), nn.Linear(784, 512), nn.LeakyReLU(0.2), nn.Linear(512, 256), nn.LeakyReLU(0.2), nn.Linear(256, 1))

class DeepConvGAN(nn.Module):
    def __init__(self, z_dim=100):
        super().__init__()
        self.generator = nn.Sequential(
            nn.Unflatten(1, (z_dim, 1, 1)),
            nn.ConvTranspose2d(z_dim, 64, 7, bias=False), nn.BatchNorm2d(64), nn.ReLU(),
            nn.ConvTranspose2d(64, 32, 4, 2, 1, bias=False), nn.BatchNorm2d(32), nn.ReLU(),
            nn.ConvTranspose2d(32, 1, 4, 2, 1), nn.Tanh()
        )
        self.discriminator = nn.Sequential(
            nn.Conv2d(1, 32, 4, 2, 1), nn.LeakyReLU(0.2),
            nn.Conv2d(32, 64, 4, 2, 1, bias=False), nn.BatchNorm2d(64), nn.LeakyReLU(0.2),
            nn.Conv2d(64, 1, 7), nn.Flatten(1)
        )
        for m in self.modules():
            if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d, nn.BatchNorm2d)):
                nn.init.normal_(m.weight, float(isinstance(m, nn.BatchNorm2d)), 0.02)
                if m.bias is not None: nn.init.zeros_(m.bias)

In [15]:
import math
from torch.nn import functional as F

class TimeBlock(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.conv, self.norm, self.time = nn.Conv2d(ch, ch, 3, padding=1), nn.GroupNorm(4, ch), nn.Linear(64, ch)
    def forward(self, x, t):
        return F.silu(self.norm(self.conv(x)) + self.time(t)[:, :, None, None])

class CosineDDPM(nn.Module):
    steps = 200
    def __init__(self):
        super().__init__()
        t = torch.linspace(0, 1, self.steps + 1, dtype=torch.float64)
        cos = ((t + 0.008) / 1.008 * math.pi / 2).cos().square()
        beta = F.pad((1 - cos[1:] / cos[:-1]).clamp(1e-8, 0.999), (1, 0))
        alpha, abar = 1 - beta, (1 - beta).cumprod(0)
        var = (1 - abar[1:]).clamp_min(1e-20)
        ang = torch.arange(self.steps + 1)[:, None] * (-math.log(10000) * torch.arange(32) / 32).exp()
        self.register_buffer("embeddings", torch.cat((ang.sin(), ang.cos()), 1).float())
        self.register_buffer("alpha_bar", abar.float())
        self.register_buffer("clean_scale", abar.sqrt().float())
        self.register_buffer("noise_scale", (1 - abar).sqrt().float())
        self.register_buffer("posterior_std", F.pad((beta[1:] * (1 - abar[:-1]) / var).sqrt(), (1, 0)).float())
        self.register_buffer("posterior_x0", F.pad(beta[1:] * abar[:-1].sqrt() / var, (1, 0)).float(), persistent=False)
        self.register_buffer("posterior_xt", F.pad(alpha[1:].sqrt() * (1 - abar[:-1]) / var, (1, 0)).float(), persistent=False)
        self.time = nn.Sequential(nn.Linear(64, 64), nn.SiLU(), nn.Linear(64, 64))
        self.stem = nn.Conv2d(1, 16, 3, padding=1)
        self.encode_28, self.encode_14, self.middle = TimeBlock(16), TimeBlock(32), TimeBlock(64)
        self.down_14, self.down_7 = nn.Conv2d(16, 32, 4, 2, 1), nn.Conv2d(32, 64, 4, 2, 1)
        self.up_14, self.up_28 = nn.ConvTranspose2d(64, 32, 4, 2, 1), nn.ConvTranspose2d(32, 16, 4, 2, 1)
        self.decode_14, self.decode_28 = TimeBlock(32), TimeBlock(16)
        self.output = nn.Conv2d(16, 1, 3, padding=1)

    def forward(self, x, step):
        t = self.time(self.embeddings[step])
        h1 = self.encode_28(self.stem(x), t)
        h2 = self.encode_14(self.down_14(h1), t)
        h = self.middle(self.down_7(h2), t)
        h = self.decode_14(self.up_14(h) + h2, t)
        return self.output(self.decode_28(self.up_28(h) + h1, t))

    def objective(self, x):
        step = torch.randint(1, self.steps + 1, (len(x),), device=x.device)
        idx = step[:, None, None, None]
        eps = torch.randn_like(x)
        return F.mse_loss(self(self.clean_scale[idx] * (2 * x - 1) + self.noise_scale[idx] * eps, step), eps)

    @torch.inference_mode()
    def sample(self, x, rng):
        for t in range(self.steps, 0, -1):
            step = torch.full((len(x),), t, dtype=torch.long, device=x.device)
            x0 = ((x - self.noise_scale[t] * self(x, step)) / self.clean_scale[t]).clamp(-1, 1)
            x = self.posterior_x0[t] * x0 + self.posterior_xt[t] * x + (self.posterior_std[t] * torch.randn(x.shape, device=x.device, generator=rng) if t > 1 else 0)
        return (x / 2 + 0.5).clamp(0, 1)